<a href="https://colab.research.google.com/github/HLZHarry/Alpha-Lens-TPM/blob/main/03_AI_Macro_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this phase, we are solving a classic institutional problem: Market data is "Lagged" (Backward-looking), but Central Bank text is "Forward-looking."

We will build an agent that reads a simulated "Federal Reserve Statement" and converts the "Hawkish" (High Interest Rate) or "Dovish" (Low Interest Rate) tone into a numerical Regime Score.

In [9]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from google.colab import userdata
from google.genai import Client
import time
from openai import OpenAI

In [16]:
# 1. Initialize the new client
# client = Client(api_key = userdata.get('GOOGLE_API_KEY'))
client = OpenAI(api_key = userdata.get('OPENAI_API_KEY'))

In [17]:
# 2. Historical Meeting Dates
meeting_dates = [
    '20210127', '20210317', '20210428', '20210616', '20210728', '20210922', '20211103', '20211215',
    '20220126', '20220316', '20220504', '20220615', '20220727', '20220921', '20221102', '20221214',
    '20230201', '20230322', '20230503', '20230614', '20230726', '20230920', '20231101', '20231213',
    '20240131', '20240320', '20240501', '20240612', '20240731', '20240918', '20241107', '20241218',
    '20250129', '20250319', '20250507', '20250618', '20250730', '20250917', '20251029', '20251210',
    '20260128'
]

results = []

In [18]:
# Helper: LLM Sentiment Agent
def get_llm_sentiment(text):
  """
  Analyzes Feb text and returns a score from -1.0 (Hawkish) to 1.0 (Dovish).
  """
  try:
    response = client.chat.completions.create(
        model = "gpt-4o", # Using GPT-4o for high-fidelity reasoning
        messages = [
            {"role": "system", "content": "You are a Senior Quantitative Macro \
            Analyst. Analyze the Fed statement for interest rate bias."},

            {"role": "user", "content": f"Based on this text, provide a sentiment \
            score from -1.0 (Hawkish/Restrictive) to 1.0 (Dovish/Accommodative). \
            Return ONLY the numerical float. Text: {text[:3000]}"}
        ],
        temperature = 0
    )
    return float(response.choices[0].message.content.strip())
  except Exception as e:
    print(f"Error calling OpenAI: {e}")
    return None

In [19]:
# 4. The Data Extraction Loop:
print("Starting Historical Macro Scraper...")

for date in meeting_dates:
  url = f"https://www.federalreserve.gov/newsevents/pressreleases/monetary{date}a.htm"
  print(f"Scraping {date}...", end = " ")

  try:
    # Scrape Text
    resp = requests.get(url)
    soup = BeautifulSoup(resp.content, 'html.parser')
    full_text = " ".join([p.text for p in soup.find_all('p') if len(p.text) > 100])

    # Analyze
    score = get_llm_sentiment(full_text)

    if score is not None:
      results.append({'Date': pd.to_datetime(date), "AI Sentiment": score})
      print(f"Score {score}")

  except Exception as e:
      print(f"Failed: {e}")

Starting Historical Macro Scraper...
Scraping 20210127... Score 0.8
Scraping 20210317... Score 0.8
Scraping 20210428... Score 0.8
Scraping 20210616... Score 0.8
Scraping 20210728... Score 0.8
Scraping 20210922... Score 0.5
Scraping 20211103... Score -0.2
Scraping 20211215... Score -0.2
Scraping 20220126... Score -0.3
Scraping 20220316... Score -0.5
Scraping 20220504... Score -0.7
Scraping 20220615... Score -0.7
Scraping 20220727... Score -0.7
Scraping 20220921... Score -0.8
Scraping 20221102... Score -0.8
Scraping 20221214... Score -0.8
Scraping 20230201... Score -0.8
Scraping 20230322... Score -0.7
Scraping 20230503... Score -0.5
Scraping 20230614... Score -0.2
Scraping 20230726... Score -0.5
Scraping 20230920... Score -0.2
Scraping 20231101... Score -0.5
Scraping 20231213... Score -0.5
Scraping 20240131... Score -0.3
Scraping 20240320... Score -0.2
Scraping 20240501... Score -0.3
Scraping 20240612... Score -0.3
Scraping 20240731... Score -0.3
Scraping 20240918... Score 0.5
Scraping 2

In [24]:
# 5. Integration with market data:
drive.mount('/content/drive')
path = "/content/drive/MyDrive/Alpha-Lens-Project"
df = pd.read_csv(f"{path}/engineered_features.csv")
df['Date'] = pd.to_datetime(df['Date'])

# Create sentiment dataframe and merge
sentiment_df = pd.DataFrame(results)
df = pd.merge(df, sentiment_df, on='Date', how='left')

# Propagation Step
# This ffill ensoures the regime persists until the next meeting
df['AI Sentiment'] = df.groupby('Ticker')['AI Sentiment'].ffill().fillna(0)
df = df.rename(columns={'AI Sentiment': 'AI_Sentiment'})

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
# 6. Save the golden dataset
df.to_csv(f"{path}/final_hybrid_data.csv", index=False)
print(f"✅ Phase 3 Complete! 'final_hybrid_data.csv' saved with {len(df)} rows.")
df[['Date', 'Ticker', 'AI_Sentiment']].tail(10)

✅ Phase 3 Complete! 'final_hybrid_data.csv' saved with 17257 rows.


,Date,Ticker,AI_Sentiment
17247,2025-12-30,GOOGL,-0.3
17248,2025-12-30,GS,-0.3
17249,2025-12-30,JNJ,-0.3
17250,2025-12-30,JPM,-0.3
17251,2025-12-30,MSFT,-0.3
17252,2025-12-30,PFE,-0.3
17253,2025-12-30,RY.TO,-0.3
17254,2025-12-30,SHOP.TO,-0.3
17255,2025-12-30,TD.TO,-0.3
17256,2025-12-30,XOM,-0.3
